In [0]:
EH_NAMESPACE = '<namespace>.servicebus.windows.net'
EH_NAME = '<event-hub-name>'
SERVICE_CREDENTIAL = '<service-credential>'

CATALOG = spark.sql('SELECT current_catalog()').first()[0]
SCHEMA = 'iss_tracker'

BRONZE = f'{SCHEMA}.bronze_events'
SILVER = f'{SCHEMA}.silver_positions'
GOLD = f'{SCHEMA}.gold_positions'

CKPT_ROOT = f'/Volumes/{CATALOG}/{SCHEMA}/pipeline_state/checkpoints'
CKPT_BRONZE = f'{CKPT_ROOT}/bronze_v1'
CKPT_SILVER = f'{CKPT_ROOT}/silver_v2'

KAFKA_OPTIONS = {
    'kafka.bootstrap.servers': f'{EH_NAMESPACE}:9093',
    'subscribe': EH_NAME,
    'databricks.serviceCredential': SERVICE_CREDENTIAL
}

print('CATALOG    :', CATALOG)
print('Hub        :', EH_NAMESPACE, '/', EH_NAME)
print('Checkpoints:', CKPT_ROOT)

In [0]:
from pyspark.sql.functions import col, get_json_object, date_format

def run_silver_stream():
    parsed = (spark.readStream.table(BRONZE)
              .select(
                  get_json_object('raw_json', '$.iss_position.latitude').cast('double').alias('lat'),
                  get_json_object('raw_json', '$.iss_position.longitude').cast('double').alias('lon'),
                  get_json_object('raw_json', '$.timestamp').cast('long').alias('event_ts'),
                  col('broker_time'),
                  col('partition'),
                  col('offset')
              )
              .filter(col('lat').between(-90, 90))
              .filter(col('lon').between(-180, 180))
              .filter(col('event_ts') > 0)
              .withColumn('event_dt', col('event_ts').cast('timestamp'))
              .withColumn('date_part', date_format('event_dt', 'yyyy-MM-dd')))
    
    query = (parsed.writeStream
             .format('delta')
             .option('checkpointLocation', CKPT_SILVER)
             .trigger(availableNow=True)
             .toTable(SILVER))
    query.awaitTermination()
    return query

In [0]:
q = run_silver_stream()
print('Query active: ', q.isActive)

bronze_n = spark.table(BRONZE).count()
silver = spark.table(SILVER)
print('Bronze rows: ', bronze_n)
print('Silver rows: ', silver.count())
silver.orderBy(col('event_ts').desc()).show(5)

In [0]:
before = spark.table(SILVER).count()
run_silver_stream()
after = spark.table(SILVER).count()
print(f'before = {before}, after = {after}, added = {after - before}')

In [0]:
from pyspark.sql.functions import count, min, max, avg

spark.table(SILVER).select(
    count('*').alias('rows'),
    min('lat').alias('min_lat'),
    max('lat').alias('max_lat'),
    avg('lat').alias('avg_lat'),
    min('event_dt').alias('first'),
    max('event_dt').alias('last')
).show(truncate=False)

spark.sql(f'DESCRIBE HISTORY {SILVER}').select('version', 'timestamp', 'operation').show(10, truncate=60)